In [1]:
# Create a Table For Nav_History

In [2]:
import pandas as pd
import plotly.express as px

nav = pd.read_csv("C:/MutualFundProject/data/raw/02_nav_history.csv")

nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [3]:
fund_master = pd.read_csv("C:/MutualFundProject/data/raw/01_fund_master.csv")

print(fund_master.columns)

Index(['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
       'plan', 'launch_date', 'benchmark', 'expense_ratio_pct',
       'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager',
       'risk_category', 'sebi_category_code'],
      dtype='str')


In [4]:
nav = nav.merge(
    fund_master[["amfi_code", "scheme_name"]],
    on="amfi_code",
    how="left"
)

In [5]:
print(nav["scheme_name"].isna().sum())

0


In [6]:
print(nav.columns)

Index(['amfi_code', 'date', 'nav', 'scheme_name'], dtype='str')


In [7]:
nav.to_csv(
    "c:/MutualFundProject/data/processed/nav_history.csv",
    index=False
)

In [8]:
## Create Table For Aum_History

In [9]:
aum = pd.read_csv("c:/MutualFundProject/data/raw/03_fund_aum.csv")

aum.head()

,date,fund_house,aum_lakh_crore,aum_crore,num_schemes
0,2022-03-31,SBI Mutual Fund,6.05,605000,186
1,2022-03-31,ICICI Prudential MF,4.65,465000,216
2,2022-03-31,HDFC Mutual Fund,4.35,435000,195
3,2022-03-31,Nippon India MF,2.70,270000,177
4,2022-03-31,Kotak Mahindra MF,2.70,270000,168


In [10]:
aum.rename(
    columns={"aum_crore": "aum"},
    inplace=True
)

In [11]:
aum.shape

(90, 5)

In [12]:
aum.dtypes

date                  str
fund_house            str
aum_lakh_crore    float64
aum                 int64
num_schemes         int64
dtype: object

In [13]:
aum["date"] = pd.to_datetime(aum["date"])

In [14]:
aum["year"] = (aum["date"].dt.year)

In [15]:
aum_history = (
    aum
    .sort_values(["fund_house", "date"])
    .groupby(["fund_house", "year"])
    .tail(1)
)

In [16]:
aum_history = aum_history[
    ["fund_house", "year", "aum"]
]

In [17]:
aum_history.head()

,fund_house,year,aum
15,Aditya Birla Sun Life MF,2022,285000
35,Aditya Birla Sun Life MF,2023,308000
65,Aditya Birla Sun Life MF,2024,384000
85,Aditya Birla Sun Life MF,2025,460000
16,Axis Mutual Fund,2022,240000


In [18]:
import os

os.makedirs("c:/MutualFundProject/data/processed", exist_ok=True)

In [19]:
aum_history.to_csv(
    "c:/MutualFundProject/data/processed/aum_history.csv",
    index=False
)

In [20]:
# Investor_demographics Table

In [21]:
import pandas as pd

df = pd.read_csv("C:/MutualFundProject/data/raw/08_investor_transactions.csv")

print(df.shape)
print(df.head())
print(df.columns)
print(df.dtypes)

(32778, 13)
  investor_id transaction_date  amfi_code transaction_type  amount_inr  \
0   INV003054       2024-01-01     119092              SIP        1834   
1   INV002952       2024-01-01     148567       Redemption      392882   
2   INV003420       2024-01-01     118636              SIP         912   
3   INV003436       2024-01-01     118634              SIP        1102   
4   INV004691       2024-01-01     119094          Lumpsum        8682   

         state       city city_tier age_group  gender  annual_income_lakh  \
0    Telangana  Hyderabad       T30       56+  Female                77.1   
1       Punjab   Amritsar       B30     18-25    Male                 7.1   
2      Haryana  Faridabad       B30     36-45    Male                47.2   
3  Maharashtra     Mumbai       T30     36-45  Female                54.4   
4        Delhi      Noida       T30     26-35    Male                14.5   

  payment_mode kyc_status  
0          UPI   Verified  
1       Cheque   Verifie

In [22]:
sip_df = df.rename(
    columns={'amount_inr': 'sip_amount'}
)

In [23]:
sip_amount_table = sip_df[
    ['investor_id', 'age_group', 'gender', 'sip_amount']
].copy()
print(sip_amount_table.head())

  investor_id age_group  gender  sip_amount
0   INV003054       56+  Female        1834
1   INV002952     18-25    Male      392882
2   INV003420     36-45    Male         912
3   INV003436     36-45  Female        1102
4   INV004691     26-35    Male        8682


In [24]:
print(sip_df[['investor_id', 'age_group', 'gender', 'sip_amount']].isnull().sum())

investor_id    0
age_group      0
gender         0
sip_amount     0
dtype: int64


In [25]:
print(df['age_group'].value_counts().sort_index())

age_group
18-25     4916
26-35    13463
36-45     8146
46-55     3779
56+       2474
Name: count, dtype: int64


In [26]:
age_counts = df['age_group'].value_counts().sort_index()

print(age_counts)

age_group
18-25     4916
26-35    13463
36-45     8146
46-55     3779
56+       2474
Name: count, dtype: int64


In [27]:
investor_df = sip_df.drop(columns=["transaction_date","amfi_code","state","city","city_tier","annual_income_lakh","payment_mode","kyc_status"])
print(investor_df.head())

  investor_id transaction_type  sip_amount age_group  gender
0   INV003054              SIP        1834       56+  Female
1   INV002952       Redemption      392882     18-25    Male
2   INV003420              SIP         912     36-45    Male
3   INV003436              SIP        1102     36-45  Female
4   INV004691          Lumpsum        8682     26-35    Male


In [28]:
investor_df.to_csv(
    "C:/MutualFundProject/data/processed/08_investor_demographics.csv",
     index=False
)
print("file saved sucessfully!")

file saved sucessfully!


In [29]:
investor_df

,investor_id,transaction_type,sip_amount,age_group,gender
0,INV003054,SIP,1834,56+,Female
1,INV002952,Redemption,392882,18-25,Male
2,INV003420,SIP,912,36-45,Male
3,INV003436,SIP,1102,36-45,Female
4,INV004691,Lumpsum,8682,26-35,Male
...,...,...,...,...,...
32773,INV003340,Lumpsum,168029,26-35,Male
32774,INV001838,SIP,2175,46-55,Male
32775,INV000074,SIP,25998,26-35,Female
32776,INV002929,SIP,459,26-35,Male


In [30]:
df

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending
...,...,...,...,...,...,...,...,...,...,...,...,...,...
32773,INV003340,2025-05-30,101207,Lumpsum,168029,Madhya Pradesh,Indore,T30,26-35,Male,22.5,Net Banking,Verified
32774,INV001838,2025-05-30,119093,SIP,2175,Uttar Pradesh,Kanpur,B30,46-55,Male,27.6,Mandate,Verified
32775,INV000074,2025-05-30,120504,SIP,25998,Rajasthan,Jaipur,T30,26-35,Female,8.4,UPI,Verified
32776,INV002929,2025-05-30,148568,SIP,459,West Bengal,Kolkata,T30,26-35,Male,13.0,Mandate,Verified


In [31]:
investor_df = investor_df.merge(df[['investor_id','state','city_tier']], on='investor_id')

In [32]:
print(investor_df)

       investor_id transaction_type  sip_amount age_group  gender      state  \
0        INV003054              SIP        1834       56+  Female  Telangana   
1        INV003054              SIP        1834       56+  Female  Telangana   
2        INV003054              SIP        1834       56+  Female  Telangana   
3        INV003054              SIP        1834       56+  Female  Telangana   
4        INV003054              SIP        1834       56+  Female  Telangana   
...            ...              ...         ...       ...     ...        ...   
274427   INV001852       Redemption      226721     26-35    Male  Telangana   
274428   INV001852       Redemption      226721     26-35    Male  Telangana   
274429   INV001852       Redemption      226721     26-35    Male  Telangana   
274430   INV001852       Redemption      226721     26-35    Male  Telangana   
274431   INV001852       Redemption      226721     26-35    Male  Telangana   

       city_tier  
0            T30  
1

In [33]:
investor_df.to_csv(
    "C:/MutualFundProject/data/processed/08_investor_demographics.csv",
     index=False
)
print("file saved sucessfully!")

file saved sucessfully!


In [29]:
# Portfolio_Holdings_table

In [31]:
import pandas as pd

holdings = pd.read_csv("C:/MutualFundProject/data/raw/09_portfolio_holdings.csv")
fund_master = pd.read_csv("C:/MutualFundProject/data/raw/01_fund_master.csv")

print(holdings.columns)
print(fund_master.columns)

Index(['amfi_code', 'stock_symbol', 'stock_name', 'sector', 'weight_pct',
       'market_value_cr', 'current_price_inr', 'portfolio_date'],
      dtype='str')
Index(['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category',
       'plan', 'launch_date', 'benchmark', 'expense_ratio_pct',
       'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager',
       'risk_category', 'sebi_category_code'],
      dtype='str')


In [23]:
holdings = holdings.merge(
    fund_master[
        ['amfi_code', 'scheme_name', 'fund_house']
    ],
    on='amfi_code',
    how='left'
)

In [24]:
print(holdings.head())

   amfi_code stock_symbol                stock_name       sector  weight_pct  \
0     119551    POWERGRID    Power Grid Corporation    Utilities       13.85   
1     119551     HDFCBANK             HDFC Bank Ltd      Banking       11.19   
2     119551       GRASIM     Grasim Industries Ltd  Diversified        9.90   
3     119551      DRREDDY  Dr. Reddy's Laboratories       Pharma        4.76   
4     119551   ASIANPAINT          Asian Paints Ltd       Paints       10.25   

   market_value_cr  current_price_inr portfolio_date  \
0           737.09            6011.08     2025-12-31   
1            88.97            1074.65     2025-12-31   
2           208.45            5964.59     2025-12-31   
3           161.32            3748.82     2025-12-31   
4           725.90            1321.45     2025-12-31   

                                 scheme_name       fund_house  
0  SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund  
1  SBI Bluechip Fund - Regular Plan - Growth  SBI Mutu

In [27]:
holdings_df = holdings.rename(columns={"fund_house":"fund_type","weight_pct":"weight"})

In [28]:
holdings_df.to_csv(
    "C:/MutualFundProject/data/processed/09_holdings_portfolio.csv",
     index=False
)
print("file saved sucessfully!")

file saved sucessfully!


In [1]:
# Nifty_100.CSV Table Make

In [2]:
!pip install yfinance


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import yfinance as yf
import pandas as pd

In [22]:
nifty100 = yf.download(
    "^CNX100",
    start="2022-01-01",
    end="2026-08-12",
    auto_adjust=False
)

[*********************100%***********************]  1 of 1 completed


In [23]:
nifty100 = nifty100.reset_index()

In [24]:
nifty100.head()

Price,Date,Adj Close,Close,High,Low,Open,Volume
Ticker,,^CNX100,^CNX100,^CNX100,^CNX100,^CNX100,^CNX100
0,2022-01-03,17874.000000,17874.000000,17894.150391,17651.449219,17654.449219,5756800
1,2022-01-04,18030.949219,18030.949219,18051.500000,17831.500000,17934.500000,6744900
2,2022-01-05,18147.500000,18147.500000,18164.699219,17974.900391,18044.300781,7898200
3,2022-01-06,17985.599609,17985.599609,18025.250000,17889.750000,17997.599609,5508600
4,2022-01-07,18053.949219,18053.949219,18140.849609,17944.400391,18035.699219,5914700


In [25]:
nifty100.tail()

Price,Date,Adj Close,Close,High,Low,Open,Volume
Ticker,,^CNX100,^CNX100,^CNX100,^CNX100,^CNX100,^CNX100
1126,2026-08-05,25744.050781,25744.050781,25801.449219,25622.849609,25794.300781,5653700
1127,2026-08-06,25757.400391,25757.400391,25798.449219,25720.050781,25758.599609,5965600
1128,2026-08-07,25712.699219,25712.699219,25771.199219,25664.250000,25672.400391,5462900
1129,2026-08-10,25728.300781,25728.300781,25761.000000,25653.300781,25726.800781,4963400
1130,2026-08-11,25625.400391,25625.400391,25729.199219,25578.300781,25728.050781,4856300


In [26]:
nifty100 = nifty100.reset_index()

nifty100 = nifty100[['Date', 'Close']]

In [27]:
nifty100.columns = ['date', 'close']

In [28]:
nifty100.head()

,date,close
0,2022-01-03,17874.000000
1,2022-01-04,18030.949219
2,2022-01-05,18147.500000
3,2022-01-06,17985.599609
4,2022-01-07,18053.949219


In [29]:
nifty100['date'] = pd.to_datetime(
    nifty100['date']
)

nifty100['close'] = pd.to_numeric(
    nifty100['close'],
    errors='coerce'
)

In [30]:
nifty100 = nifty100.dropna(
    subset=['date', 'close']
)

In [31]:
nifty100['nifty100_return'] = (
    nifty100['close'].pct_change()
)

In [32]:
nifty100.head()

,date,close,nifty100_return
0,2022-01-03,17874.000000,NaN
1,2022-01-04,18030.949219,0.008781
2,2022-01-05,18147.500000,0.006464
3,2022-01-06,17985.599609,-0.008921
4,2022-01-07,18053.949219,0.003800


In [33]:
nifty100 = nifty100.dropna(
    subset=['nifty100_return']
)

In [34]:
nifty100.to_csv(
    "C:/MutualFundProject/data/processed/nifty100_daily.csv",
    index=False
)

In [55]:
# Nifty_50.CSV Table

In [56]:
import yfinance as yf
import pandas as pd

In [57]:
nifty50 = yf.download(
    "^NSEI",
    start="2022-01-01",
    end="2026-08-12",
    auto_adjust=False
)

[*********************100%***********************]  1 of 1 completed


In [58]:
nifty50 = nifty50.reset_index()

In [59]:
nifty50.head()

Price,Date,Adj Close,Close,High,Low,Open,Volume
Ticker,,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
0,2022-01-03,17625.699219,17625.699219,17646.650391,17383.300781,17387.150391,200500
1,2022-01-04,17805.250000,17805.250000,17827.599609,17593.550781,17681.400391,247400
2,2022-01-05,17925.250000,17925.250000,17944.699219,17748.849609,17820.099609,251500
3,2022-01-06,17745.900391,17745.900391,17797.949219,17655.550781,17768.500000,236500
4,2022-01-07,17812.699219,17812.699219,17905.000000,17704.550781,17797.599609,239300


In [60]:
nifty50.tail()

Price,Date,Adj Close,Close,High,Low,Open,Volume
Ticker,,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
1132,2026-08-05,24624.650391,24624.650391,24677.599609,24497.949219,24669.199219,355500
1133,2026-08-06,24636.000000,24636.000000,24677.050781,24604.150391,24641.000000,344000
1134,2026-08-07,24570.650391,24570.650391,24630.400391,24522.750000,24538.900391,254800
1135,2026-08-10,24583.800781,24583.800781,24620.949219,24511.099609,24581.250000,269500
1136,2026-08-11,24471.699219,24471.699219,24576.849609,24429.250000,24575.099609,271800


In [61]:
nifty50 = nifty50.reset_index()

nifty50 = nifty50[['Date', 'Close']]

In [62]:
nifty50.columns = ['date', 'close']

In [63]:
nifty50.head()

,date,close
0,2022-01-03,17625.699219
1,2022-01-04,17805.250000
2,2022-01-05,17925.250000
3,2022-01-06,17745.900391
4,2022-01-07,17812.699219


In [65]:
nifty50['nifty100_return'] = (
    nifty50['close'].pct_change()
)

In [66]:
nifty50.head()

,date,close,nifty100_return
0,2022-01-03,17625.699219,NaN
1,2022-01-04,17805.250000,0.010187
2,2022-01-05,17925.250000,0.006740
3,2022-01-06,17745.900391,-0.010005
4,2022-01-07,17812.699219,0.003764


In [68]:
nifty50 = nifty50.reset_index()

nifty50 = nifty50[['date', 'close']]

In [72]:
nifty50.columns = ['date', 'close']

In [73]:
nifty50.head()

,date,close
0,2022-01-03,17625.699219
1,2022-01-04,17805.250000
2,2022-01-05,17925.250000
3,2022-01-06,17745.900391
4,2022-01-07,17812.699219


In [74]:
nifty50['date'] = pd.to_datetime(
    nifty50['date']
)

nifty50['close'] = pd.to_numeric(
    nifty50['close'],
    errors='coerce'
)

In [75]:
nifty50 = nifty50.dropna(
    subset=['date', 'close']
)

In [76]:
nifty50['nifty50_return'] = (
    nifty50['close'].pct_change()
)

In [77]:
nifty50.head()

,date,close,nifty50_return
0,2022-01-03,17625.699219,NaN
1,2022-01-04,17805.250000,0.010187
2,2022-01-05,17925.250000,0.006740
3,2022-01-06,17745.900391,-0.010005
4,2022-01-07,17812.699219,0.003764


In [78]:
nifty50 = nifty50.dropna(
    subset=['nifty50_return']
)

In [79]:
nifty50.to_csv(
    "C:/MutualFundProject/data/processed/nifty50_daily.csv",
     index=False
)